In [2]:
import sqlite3, time
import pandas as pd

DB = "../data/processed/steam.db"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

bas = time.time()
sorgu = """
SELECT
  COUNT(*)                               AS yorum_sayisi,
  SUM(LENGTH(metin))                     AS toplam_karakter,
  AVG(LENGTH(metin))                     AS ort_karakter,
  SUM(LENGTH(metin) < 100)               AS kisa_100_alti,
  SUM(LENGTH(metin) BETWEEN 100 AND 499) AS orta_100_499,
  SUM(LENGTH(metin) BETWEEN 500 AND 1999) AS uzun_500_1999,
  SUM(LENGTH(metin) >= 2000)             AS cok_uzun_2000_ustu
FROM yorumlar
WHERE chroma_uygun = 1
"""
ozet = pd.read_sql(sorgu, con)
print(f"Süre: {time.time() - bas:.1f} sn")
print(ozet.T.to_string())

Süre: 33.0 sn
                               0
yorum_sayisi        2.184677e+07
toplam_karakter     6.591193e+09
ort_karakter        3.017011e+02
kisa_100_alti       1.073101e+07
orta_100_499        7.898894e+06
uzun_500_1999       2.714960e+06
cok_uzun_2000_ustu  5.019010e+05


## 1. Yorum hacmi (chroma_uygun = 1)
Kaynak: steam.db `yorumlar` tablosu, sorgu 33 sn.

| Ölçüm | Değer |
|---|---|
| Uygun yorum | 21,85 M |
| Toplam karakter | 6,59 milyar |
| Ortalama | 302 karakter (~75 token, 4 kar./token kabaca) |
| < 100 kar. | 10,73 M (%49) |
| 100–499 | 7,90 M (%36) |
| 500–1999 | 2,71 M (%12) |
| ≥ 2000 | 0,50 M (%2) |

- Yorumların yarısı çok kısa → az bilgi, örnekleme kararında dikkate alınacak.
- 75 token kaba tahmin; diller farklı bölündüğü için gerçek token sayısı ayrıca ölçülecek.

In [3]:
import random
from transformers import AutoTokenizer

random.seed(42)
en_buyuk = con.execute("SELECT MAX(rowid) FROM yorumlar").fetchone()[0]
rastgele_idler = random.sample(range(1, en_buyuk + 1), 30_000)

id_listesi = ",".join(map(str, rastgele_idler))
ornek = pd.read_sql(
    f"SELECT review_id, app_id, dil, metin FROM yorumlar "
    f"WHERE rowid IN ({id_listesi}) AND chroma_uygun = 1",
    con,
)
print("Örnek yorum sayısı:", len(ornek))

tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
ornek["karakter"] = ornek["metin"].str.len()
ornek["token"] = [len(x) for x in tok(ornek["metin"].tolist(), add_special_tokens=False)["input_ids"]]

print("Ortalama karakter:", round(ornek["karakter"].mean(), 1))
print("Ortalama token   :", round(ornek["token"].mean(), 1))
print("Karakter / token :", round(ornek["karakter"].sum() / ornek["token"].sum(), 2))
print()

dil_ozet = (
    ornek.groupby("dil")
    .agg(yorum=("token", "size"), ort_token=("token", "mean"), ort_karakter=("karakter", "mean"))
    .sort_values("yorum", ascending=False)
    .round(1)
)
print(dil_ozet.head(15).to_string())

/home/kaan/steam-oyun-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Örnek yorum sayısı: 20737


Ortalama karakter: 301.0
Ortalama token   : 88.0
Karakter / token : 3.42

            yorum  ort_token  ort_karakter
dil                                       
en          11274       84.6         339.5
zh           2928       97.0         139.9
ru           1665       92.3         327.9
es            829       69.9         284.6
pt            671       66.8         271.0
de            613       97.5         414.3
fr            478      120.3         458.6
bilinmiyor    463       72.6         168.6
tr            458       54.2         217.8
ko            440      114.9         202.5
ja            273      174.3         318.8
pl            265       73.0         270.3
it            120      164.6         681.3
uk             69       90.7         314.0
cs             46       86.8         306.3


## 2. Gerçek token ölçümü (bge-m3 tokenizer)
Örnek: rastgele 30.000 rowid (seed 42) → chroma_uygun 20.737 yorum.
Örnek ortalaması 301 karakter ≈ bütün veri 302 → örnek temsil ediyor.

- Ortalama **88 token** (kaba tahmin 75 idi), **3,42 karakter/token**.
- Diller farklı bölünüyor: zh 140 kar. → 97 token, ja 319 → 174, tr 218 → 54, en 340 → 85.
- Tahmini toplam: 88 × 21,85 M ≈ **1,92 milyar token**.

| | Token | $0,20/1M | Batch $0,10/1M |
|---|---|---|---|
| 1 M yorum | 88 M | ~$18 | ~$9 |
| Hepsi | 1,92 milyar | ~$385 | ~$192 |

Not: Gemini tokenizer farklı sayabilir; kalite testinde küçük örnekle kontrol edilecek.
Maliyeti düşürecek iki kol: örnekleme ve uzun yorumu kesme.

In [4]:
toplam_token = ornek["token"].sum()

satirlar = []
for sinir in [64, 128, 256, 512, 1024]:
    kesilen_yorum = (ornek["token"] > sinir).mean() * 100
    kalan_token = ornek["token"].clip(upper=sinir).sum()
    satirlar.append({
        "sinir": sinir,
        "kesilen_yorum_%": round(kesilen_yorum, 1),
        "kalan_token_%": round(kalan_token / toplam_token * 100, 1),
        "ort_token": round(kalan_token / len(ornek), 1),
    })

kesme = pd.DataFrame(satirlar)
kesme["tahmini_toplam_milyon_token"] = (kesme["ort_token"] * 21.85).round(0)
print(kesme.to_string(index=False))

 sinir  kesilen_yorum_%  kalan_token_%  ort_token  tahmini_toplam_milyon_token
    64             30.8           41.7       36.7                        802.0
   128             16.6           58.2       51.2                       1119.0
   256              7.5           74.3       65.4                       1429.0
   512              2.7           87.6       77.1                       1685.0
  1024              0.7           96.1       84.5                       1846.0


## 3. Kol 1: Uzun yorumu kesme (embedding metni için; tam metin SQL'de kalır)
| Sınır | Kesilen yorum | Kalan token | Ort. token | Toplam (milyon) |
|---|---|---|---|---|
| 64 | %30,8 | %41,7 | 36,7 | 802 |
| 128 | %16,6 | %58,2 | 51,2 | 1.119 |
| 256 | %7,5 | %74,3 | 65,4 | 1.429 |
| 512 | %2,7 | %87,6 | 77,1 | 1.685 |
| 1024 | %0,7 | %96,1 | 84,5 | 1.846 |

- Az sayıda uzun yorum token'ların büyük kısmını tutuyor (512'de %2,7 yorum → %12 token).
- Kesme **maliyeti** düşürür, **depolamayı düşürmez** (her yorum = 1 vektör). Depolama için örnekleme gerekir.
- $10 bütçe: kesmeyle bile hepsi sığmıyor (3-small 512'de ~$17) → örnekleme şart.
- Aday sınır: 512 (kesin karar kalite testinden sonra).

In [5]:
oyun_sayilari = pd.read_sql(
    "SELECT app_id, ad, chroma_yorum_sayisi FROM oyunlar WHERE chroma_yorum_sayisi > 0",
    con,
)
n = oyun_sayilari["chroma_yorum_sayisi"]
print("Yorumlu oyun:", len(n), "| toplam yorum:", n.sum())
print(n.describe(percentiles=[.5, .75, .9, .99]).round(0).to_string())
print()

satirlar = []
for tavan in [100, 250, 500, 1000, 2000, 5000]:
    toplam = n.clip(upper=tavan).sum()
    token_m = toplam * 77.1 / 1e6
    satirlar.append({
        "tavan": tavan,
        "etkilenen_oyun": int((n > tavan).sum()),
        "toplam_yorum_milyon": round(toplam / 1e6, 2),
        "token_milyon": round(token_m),
        "3small_batch_$": round(token_m * 0.01, 1),
        "gemini_batch_$": round(token_m * 0.10, 1),
        "depolama_GB": round(toplam * 1024 * 4 / 1e9, 1),
    })
print(pd.DataFrame(satirlar).to_string(index=False))

Yorumlu oyun: 22898 | toplam yorum: 21846768
count     22898.0
mean        954.0
std        5272.0
min           5.0
50%          83.0
75%         294.0
90%        1280.0
99%       17177.0
max      255139.0

 tavan  etkilenen_oyun  toplam_yorum_milyon  token_milyon  3small_batch_$  gemini_batch_$  depolama_GB
   100           10328                 1.62           125             1.2            12.5          6.6
   250            6261                 2.79           215             2.2            21.5         11.4
   500            4176                 4.05           312             3.1            31.2         16.6
  1000            2700                 5.70           440             4.4            44.0         23.4
  2000            1672                 7.80           601             6.0            60.1         31.9
  5000             814                11.22           865             8.7            86.5         46.0


## 4. Kol 2: Oyun başına tavan
Kaynak: oyunlar.chroma_yorum_sayisi (toplam 21.846.768 = 1. hücre ✓), 22.898 oyun.

Dağılım: medyan 83, %75 294, %90 1.280, %99 17.177, max 255.139, ortalama 954
→ yorumlar birkaç dev oyunda; tavan küçük oyunlara dokunmaz.

| Tavan | Etkilenen oyun | Yorum (M) | Token (M, 512 kesimli) | 3-small batch | Gemini batch | Depolama (1024 boyut) |
|---|---|---|---|---|---|---|
| 100 | 10.328 | 1,62 | 125 | $1,2 | $12,5 | 6,6 GB |
| 250 | 6.261 | 2,79 | 215 | $2,2 | $21,5 | 11,4 GB |
| 500 | 4.176 | 4,05 | 312 | $3,1 | $31,2 | 16,6 GB |
| 1000 | 2.700 | 5,70 | 440 | $4,4 | $44,0 | 23,4 GB |
| 2000 | 1.672 | 7,80 | 601 | $6,0 | $60,1 | 31,9 GB |
| 5000 | 814 | 11,22 | 865 | $8,7 | $86,5 | 46,0 GB |

- $10 bütçe: 3-small her satıra sığar; Gemini hiçbir satıra sığmaz; yerel modeller $0 (süre sınırlı).
- Agent soru başına 10–20 yorum çeker → aday tavan 500–1000 (karar model seçimiyle).

In [6]:
TAVAN = 500
test_oyunlari = [1222670, 239140]   # Sims 4, Dying Light

def secim_karsilastir(app_id, tavan=TAVAN):
    oyun = pd.read_sql(
        "SELECT ad, olumlu_oran, yorum_kapsami FROM oyunlar WHERE app_id = ?",
        con, params=(app_id,),
    ).iloc[0]
    y = pd.read_sql(
        "SELECT olumlu, faydali_oy, tarih, uzunluk FROM yorumlar "
        "WHERE app_id = ? AND chroma_uygun = 1",
        con, params=(app_id,),
    )

    rastgele = y.sample(tavan, random_state=42)
    faydali = y.nlargest(tavan, "faydali_oy")

    olumlular = y[y["olumlu"] == 1]
    olumsuzlar = y[y["olumlu"] == 0]
    n_olumlu = min(round(tavan * oyun["olumlu_oran"]), len(olumlular))
    dengeli = pd.concat([
        olumlular.sample(n_olumlu, random_state=42),
        olumsuzlar.sample(tavan - n_olumlu, random_state=42),
    ])

    secimler = {"hepsi": y, "rastgele": rastgele, "faydali": faydali, "dengeli": dengeli}
    tablo = pd.DataFrame({
        ad: {
            "yorum": len(s),
            "olumlu_%": round(s["olumlu"].mean() * 100, 1),
            "ort_uzunluk": round(s["uzunluk"].mean()),
            "medyan_faydali_oy": s["faydali_oy"].median(),
            "2024_ve_sonrasi_%": round((s["tarih"] >= "2024-01-01").mean() * 100, 1),
        }
        for ad, s in secimler.items()
    }).T

    print(f"=== {oyun['ad']} | games.json oranı: %{oyun['olumlu_oran'] * 100:.1f} "
          f"| yorum_kapsami: {oyun['yorum_kapsami']:.3f}")
    print(tablo.to_string())
    print()

for app_id in test_oyunlari:
    secim_karsilastir(app_id)

=== The Sims™ 4 | games.json oranı: %87.4 | yorum_kapsami: 0.060
           yorum  olumlu_%  ort_uzunluk  medyan_faydali_oy  2024_ve_sonrasi_%
hepsi     7181.0      54.3        300.0                7.0               16.3
rastgele   500.0      52.2        313.0                7.0               16.2
faydali    500.0      66.2        488.0              243.5               10.2
dengeli    500.0      87.4        327.0                9.0               14.8

=== Dying Light | games.json oranı: %95.2 | yorum_kapsami: 0.413
             yorum  olumlu_%  ort_uzunluk  medyan_faydali_oy  2024_ve_sonrasi_%
hepsi     109785.0      88.9        272.0                1.0               20.3
rastgele     500.0      89.2        279.0                1.0               19.4
faydali      500.0      93.2        821.0              247.0                4.8
dengeli      500.0      95.2        261.0                1.0               21.4



## 5. Hangi yorumlar? (tavan 500, iki oyun)
| Oyun | games.json | hepsi (CSV) | rastgele | en faydalı | dengeli |
|---|---|---|---|---|---|
| Sims 4 (kapsam 0,06) | %87,4 | %54,3 | %52,2 | %66,2 | %87,4 |
| Dying Light (kapsam 0,41) | %95,2 | %88,9 | %89,2 | %93,2 | %95,2 |

- Rastgele: CSV yanlılığını kopyalar (Sims 4 sorunu Chroma'ya taşınır).
- En faydalı: uzun/bilgili (DL 821 kar. vs 272) ama eski yorumlara kayar (DL 2024+ %4,8 vs %20,3), oranı tutmaz, meme riski.
- Dengeli: oranı games.json'a eşitler, uzunluk ve güncellik "hepsi"ye yakın kalır.

In [7]:
def katman_karma(katman, n):
    """Bir katmandan n yorum: yarısı en faydalı, yarısı kalanlardan rastgele."""
    if len(katman) <= n:
        return katman
    n_faydali = n // 2
    faydali = katman.nlargest(n_faydali, "faydali_oy")
    kalan = katman.drop(faydali.index)
    rastgele = kalan.sample(n - n_faydali, random_state=42)
    return pd.concat([faydali, rastgele])


def karma_sec(y, tavan, olumlu_oran):
    """Oyunun yorumlarından en fazla `tavan` tane: olumlu/olumsuz kotası games.json oranına göre."""
    if len(y) <= tavan:
        return y
    if pd.isna(olumlu_oran):
        olumlu_oran = y["olumlu"].mean()
    olumlular = y[y["olumlu"] == 1]
    olumsuzlar = y[y["olumlu"] == 0]
    n_olumlu = min(round(tavan * olumlu_oran), len(olumlular))
    n_olumsuz = min(tavan - n_olumlu, len(olumsuzlar))
    return pd.concat([katman_karma(olumlular, n_olumlu), katman_karma(olumsuzlar, n_olumsuz)])


def secim_karsilastir2(app_id, tavan=TAVAN):
    oyun = pd.read_sql(
        "SELECT ad, olumlu_oran, yorum_kapsami FROM oyunlar WHERE app_id = ?",
        con, params=(app_id,),
    ).iloc[0]
    y = pd.read_sql(
        "SELECT olumlu, faydali_oy, tarih, uzunluk FROM yorumlar "
        "WHERE app_id = ? AND chroma_uygun = 1",
        con, params=(app_id,),
    )

    olumlular = y[y["olumlu"] == 1]
    olumsuzlar = y[y["olumlu"] == 0]
    n_olumlu = min(round(tavan * oyun["olumlu_oran"]), len(olumlular))
    secimler = {
        "hepsi": y,
        "rastgele": y.sample(tavan, random_state=42),
        "faydali": y.nlargest(tavan, "faydali_oy"),
        "dengeli": pd.concat([
            olumlular.sample(n_olumlu, random_state=42),
            olumsuzlar.sample(tavan - n_olumlu, random_state=42),
        ]),
        "karma": karma_sec(y, tavan, oyun["olumlu_oran"]),
    }
    tablo = pd.DataFrame({
        ad: {
            "yorum": len(s),
            "olumlu_%": round(s["olumlu"].mean() * 100, 1),
            "ort_uzunluk": round(s["uzunluk"].mean()),
            "medyan_faydali_oy": s["faydali_oy"].median(),
            "2024_ve_sonrasi_%": round((s["tarih"] >= "2024-01-01").mean() * 100, 1),
        }
        for ad, s in secimler.items()
    }).T

    print(f"=== {oyun['ad']} | games.json oranı: %{oyun['olumlu_oran'] * 100:.1f} "
          f"| yorum_kapsami: {oyun['yorum_kapsami']:.3f}")
    print(tablo.to_string())
    print()

for app_id in test_oyunlari:
    secim_karsilastir2(app_id)

=== The Sims™ 4 | games.json oranı: %87.4 | yorum_kapsami: 0.060
           yorum  olumlu_%  ort_uzunluk  medyan_faydali_oy  2024_ve_sonrasi_%
hepsi     7181.0      54.3        300.0                7.0               16.3
rastgele   500.0      52.2        313.0                7.0               16.2
faydali    500.0      66.2        488.0              243.5               10.2
dengeli    500.0      87.4        327.0                9.0               14.8
karma      500.0      87.4        377.0              181.5               15.2

=== Dying Light | games.json oranı: %95.2 | yorum_kapsami: 0.413
             yorum  olumlu_%  ort_uzunluk  medyan_faydali_oy  2024_ve_sonrasi_%
hepsi     109785.0      88.9        272.0                1.0               20.3
rastgele     500.0      89.2        279.0                1.0               19.4
faydali      500.0      93.2        821.0              247.0                4.8
dengeli      500.0      95.2        261.0                1.0               21.4
k

## 6. Karma seçim (stratify + yarı faydalı / yarı rastgele)
Fonksiyonlar: `katman_karma(katman, n)`, `karma_sec(y, tavan, olumlu_oran)` (tam veride kullanılacak).

| Oyun | Seçim | Olumlu % | Ort. uzunluk | Medyan faydalı oy | 2024+ % |
|---|---|---|---|---|---|
| Sims 4 | hepsi | 54,3 | 300 | 7 | 16,3 |
| Sims 4 | dengeli | 87,4 | 327 | 9 | 14,8 |
| Sims 4 | **karma** | **87,4** | 377 | 181,5 | 15,2 |
| Dying Light | hepsi | 88,9 | 272 | 1 | 20,3 |
| Dying Light | faydalı | 93,2 | 821 | 247 | 4,8 |
| Dying Light | dengeli | 95,2 | 261 | 1 | 21,4 |
| Dying Light | **karma** | **95,2** | 562 | 170 | 12,8 |

- Karma: oran games.json'a eşit, yorumlar daha bilgili, güncellik iki yolun ortasında (DL %12,8).
- Bedel: daha uzun yorum → biraz daha fazla token (bütçe sorunu değil).
- Güncellik önemli çıkarsa faydalı payı düşürülebilir (ör. 1/3).

In [8]:
TEST_OYUNLARI = [239140, 1222670, 1086940, 1091500, 1245620]
# Dying Light, Sims 4, Baldur's Gate 3, Cyberpunk 2077, ELDEN RING
TEST_TAVAN = 400

parcalar = []
for app_id in TEST_OYUNLARI:
    oyun = pd.read_sql(
        "SELECT ad, olumlu_oran FROM oyunlar WHERE app_id = ?", con, params=(app_id,)
    )
    if oyun.empty:
        print(app_id, "→ oyunlar tablosunda yok, atlandı")
        continue
    y = pd.read_sql(
        "SELECT review_id, app_id, olumlu, faydali_oy, tarih, dil, metin FROM yorumlar "
        "WHERE app_id = ? AND chroma_uygun = 1",
        con, params=(app_id,),
    )
    secilen = karma_sec(y, TEST_TAVAN, oyun["olumlu_oran"].iloc[0])
    print(f"{oyun['ad'].iloc[0]:<35} uygun: {len(y):>7}  seçilen: {len(secilen)}  "
          f"olumlu: %{secilen['olumlu'].mean() * 100:.1f}")
    parcalar.append(secilen)

test_yorumlar = pd.concat(parcalar, ignore_index=True)

kesik = tok(test_yorumlar["metin"].tolist(), add_special_tokens=False,
            truncation=True, max_length=512)["input_ids"]
test_yorumlar["embed_metin"] = tok.batch_decode(kesik)
test_yorumlar["kesildi"] = test_yorumlar["embed_metin"].str.len() < test_yorumlar["metin"].str.len() - 5

print()
print("Toplam:", len(test_yorumlar), "| kesilen:", test_yorumlar["kesildi"].sum())
print(test_yorumlar["dil"].value_counts().head(10).to_string())

test_yorumlar.to_parquet("../data/processed/kalite_testi_yorumlar.parquet", index=False)

Dying Light                         uygun:  109785  seçilen: 400  olumlu: %95.2
The Sims™ 4                         uygun:    7181  seçilen: 400  olumlu: %87.5
Baldur's Gate 3                     uygun:   40487  seçilen: 400  olumlu: %96.8
Cyberpunk 2077                      uygun:   21884  seçilen: 400  olumlu: %83.0
ELDEN RING                          uygun:   10504  seçilen: 400  olumlu: %92.5

Toplam: 2000 | kesilen: 275
dil
en            878
zh            490
ru            215
bilinmiyor     71
pt             70
es             64
tr             61
pl             31
ko             29
fr             26


In [9]:
kodlama = tok(test_yorumlar["metin"].tolist(), add_special_tokens=False,
              return_offsets_mapping=True)
test_yorumlar["token"] = [len(x) for x in kodlama["input_ids"]]
test_yorumlar["kesildi"] = test_yorumlar["token"] > 512

def ilk_512(metin, ofsetler):
    if len(ofsetler) <= 512:
        return metin
    son_karakter = ofsetler[511][1]
    return metin[:son_karakter]

test_yorumlar["embed_metin"] = [
    ilk_512(m, o) for m, o in zip(test_yorumlar["metin"], kodlama["offset_mapping"])
]

kesilmeyen = test_yorumlar[~test_yorumlar["kesildi"]]
print("Kesilen (token > 512):", test_yorumlar["kesildi"].sum())
print("Kesilmeyenlerde metin aynen korundu mu:", (kesilmeyen["embed_metin"] == kesilmeyen["metin"]).all())
print()
print(test_yorumlar.groupby("kesildi")[["token", "faydali_oy"]].median())

test_yorumlar.to_parquet("../data/processed/kalite_testi_yorumlar.parquet", index=False)

Kesilen (token > 512): 225
Kesilmeyenlerde metin aynen korundu mu: True

          token  faydali_oy
kesildi                    
False      34.0       105.0
True     1066.0       762.0


## 7. Kalite testi havuzu
5 oyun × 400 karma yorum = 2.000 (DL, Sims 4, BG3, Cyberpunk, ELDEN RING; oranlar games.json'a eşit).
Diller: en 878, zh 490, ru 215, tr 61 → Türkçe soru ↔ yabancı dil yorum testi mümkün.
Dosya: data/processed/kalite_testi_yorumlar.parquet

**512 token kesme:** `offset_mapping` ile orijinal metin 512. token'ın bittiği karakterden kesiliyor
(`batch_decode` satır sonlarını sildiği için 50 yorumu yanlışlıkla "kesildi" saymıştı).
- Kesilen 225 (%11). Medyan: kesilmeyen 34 token / faydalı oy 105, kesilen 1.066 token / faydalı oy 762.
- Faydalı yorumlar uzun inceleme yazıları → karma seçimde kesme oranı %2,7'den yüksek. Tam metin SQL'de.

In [10]:
PARCA = 512
ORTUSME = 64

def parcala(metin, ofsetler):
    n = len(ofsetler)
    if n <= PARCA:
        return [metin]
    parcalar = []
    bas = 0
    while True:
        bitis = bas + PARCA
        if n - bitis < ORTUSME:
            bitis = n
        parcalar.append(metin[ofsetler[bas][0]:ofsetler[bitis - 1][1]])
        if bitis == n:
            break
        bas = bitis - ORTUSME
    return parcalar

satirlar = []
for satir, ofsetler in zip(test_yorumlar.itertuples(), kodlama["offset_mapping"]):
    parcalar = parcala(satir.metin, ofsetler)
    for i, p in enumerate(parcalar):
        satirlar.append({
            "parca_id": f"{satir.review_id}#{i}",
            "review_id": satir.review_id,
            "app_id": satir.app_id,
            "olumlu": satir.olumlu,
            "dil": satir.dil,
            "tarih": satir.tarih,
            "faydali_oy": satir.faydali_oy,
            "parca_no": i,
            "parca_sayisi": len(parcalar),
            "embed_metin": p,
        })
test_parcalar = pd.DataFrame(satirlar)

test_parcalar["token"] = [len(x) for x in tok(test_parcalar["embed_metin"].tolist(),
                                               add_special_tokens=False)["input_ids"]]
print("Yorum:", test_yorumlar["review_id"].nunique(), "→ parça:", len(test_parcalar))
print("En uzun parça (token):", test_parcalar["token"].max())
print()
print("Yorum başına parça sayısı:")
print(test_parcalar.drop_duplicates("review_id")["parca_sayisi"].value_counts().sort_index().to_string())

ornek_id = test_parcalar.loc[test_parcalar["parca_sayisi"] == 3, "review_id"].iloc[0]
for _, p in test_parcalar[test_parcalar["review_id"] == ornek_id].iterrows():
    print(f"\n--- {p['parca_id']} ({p['token']} token) ---")
    print(p["embed_metin"])

test_parcalar.to_parquet("../data/processed/kalite_testi_parcalar.parquet", index=False)

Yorum: 2000 → parça: 2447
En uzun parça (token): 575

Yorum başına parça sayısı:
parca_sayisi
1     1792
2       92
3       46
4       42
5       19
6        3
7        2
8        1
9        2
12       1

--- 239140_72#0 (512 token) ---
Glad to see the misinformed reviews have been thrown out of from the top page. For those who have said that Dying Light's day/night mechanic is not what they thought and that time doesn't actually change in the game clearly haven't played past the intro missions. The first two missions are obviously there to get you started, so they're not going to thrust a timed day/night cycle before you get the hang of the environment and the mechanics, and thus in the intro missions, time does not change if you don't progress the quest. Past those two missions, day/night becomes very real, and incredibly terrifying. /endRantGameplay:I'm sure it's been said before, but this is what Dead Island should have been. Everything from the parkour to the combat just feels inc

## 8. Parçalama (kesme yerine)
Kural: 512 token parça, 64 token örtüşme, sona < 64 token kalırsa önceki parçaya eklenir (maks 575).
Metin `offset_mapping` ile orijinalden kesilir. parca_id = "{review_id}#{i}", metadata: review_id, app_id, olumlu, dil, tarih, faydali_oy, parca_no, parca_sayisi.

- 2.000 yorum → **2.447 parça (+%22)**; 1.792 tek parça, 208 çok parçalı (çoğu 2–4, en fazla 12). En uzun parça 575 ✓.
- Örnek 239140_72 (3 parça): #1 "Technical" bölümünü içeriyor (performans, takılma) → 512'de kesseydik kaybolacaktı.
- Parçalar cümle ortasından başlayabilir; örtüşme sınır cümlesini korur. Oyun adı parçada geçmeyebilir → app_id metadata ile filtre.
Dosya: data/processed/kalite_testi_parcalar.parquet

In [11]:
TEST_SORULARI = [
    # Dying Light
    {"id": "S01", "app_id": 239140, "soru": "Dying Light'ta gece nasıl, korkutucu mu?",
     "olcut": "Gece döngüsü, gece çıkan zombiler veya gecenin verdiği korku/gerilimden bahseder."},
    {"id": "S02", "app_id": 239140, "soru": "Oyun kasıyor mu, performansı nasıl?",
     "olcut": "FPS, takılma, optimizasyon veya donanımda nasıl çalıştığından bahseder."},
    # Sims 4
    {"id": "S03", "app_id": 1222670, "soru": "Sims 4'te DLC'ler çok mu pahalı?",
     "olcut": "DLC / genişleme paketlerinin fiyatı veya oyunun para tuzağı olmasından bahseder."},
    {"id": "S04", "app_id": 1222670, "soru": "Oyun ücretsiz olduktan sonra oynamaya değer mi?",
     "olcut": "Ana oyunun ücretsiz (free to play) olmasından bahseder."},
    # Baldur's Gate 3
    {"id": "S05", "app_id": 1086940, "soru": "Baldur's Gate 3 arkadaşlarla co-op oynanır mı?",
     "olcut": "Çok oyunculu / co-op deneyiminden bahseder."},
    {"id": "S06", "app_id": 1086940, "soru": "Seçimlerim hikayeyi gerçekten değiştiriyor mu?",
     "olcut": "Seçimlerin sonuçları, hikayenin dallanması veya rol yapma özgürlüğünden bahseder."},
    {"id": "S07", "app_id": 1086940, "soru": "D&D bilmeyen biri oynayabilir mi?",
     "olcut": "Yeni başlayanlar, D&D kuralları, öğrenme zorluğu veya sıra tabanlı savaşa alışmaktan bahseder."},
    # Cyberpunk 2077
    {"id": "S08", "app_id": 1091500, "soru": "Cyberpunk çıktığında çok hatalıydı, şimdi düzeldi mi?",
     "olcut": "Çıkıştaki hatalardan ve/veya güncellemelerle (yamalar, 2.0) düzelmesinden bahseder."},
    {"id": "S09", "app_id": 1091500, "soru": "Night City'nin atmosferi nasıl?",
     "olcut": "Şehrin, dünyanın veya atmosferin kendisinden bahseder."},
    {"id": "S10", "app_id": 1091500, "soru": "Oyun sürekli çöküyor mu?",
     "olcut": "Çökme, masaüstüne atma veya oyunun kapanmasından bahseder (genel hatalar yetmez)."},
    {"id": "S11", "app_id": 1091500, "soru": "Hemen mi alayım, indirimi mi bekleyeyim?",
     "olcut": "Fiyat, fiyatına değip değmediği veya indirimde almaktan bahseder."},
    # ELDEN RING
    {"id": "S12", "app_id": 1245620, "soru": "Elden Ring yeni başlayan için çok mu zor?",
     "olcut": "Zorluk, ölme, souls oyunu deneyimi olmayanlar veya öğrenme eğrisinden bahseder."},
    {"id": "S13", "app_id": 1245620, "soru": "Açık dünyayı keşfetmek keyifli mi?",
     "olcut": "Keşif, açık dünya tasarımı veya gizli yerler bulmaktan bahseder."},
    # Bütün havuz
    {"id": "S14", "app_id": None, "soru": "Türkçe dil desteği var mı?",
     "olcut": "Türkçe dil, altyazı, çeviri veya yerelleştirmeden bahseder."},
]

test_sorulari = pd.DataFrame(TEST_SORULARI)
print(test_sorulari[["id", "app_id", "soru"]].to_string(index=False))
test_sorulari.to_parquet("../data/processed/kalite_testi_sorular.parquet", index=False)

 id    app_id                                                  soru
S01  239140.0              Dying Light'ta gece nasıl, korkutucu mu?
S02  239140.0                   Oyun kasıyor mu, performansı nasıl?
S03 1222670.0                      Sims 4'te DLC'ler çok mu pahalı?
S04 1222670.0       Oyun ücretsiz olduktan sonra oynamaya değer mi?
S05 1086940.0        Baldur's Gate 3 arkadaşlarla co-op oynanır mı?
S06 1086940.0        Seçimlerim hikayeyi gerçekten değiştiriyor mu?
S07 1086940.0                     D&D bilmeyen biri oynayabilir mi?
S08 1091500.0 Cyberpunk çıktığında çok hatalıydı, şimdi düzeldi mi?
S09 1091500.0                       Night City'nin atmosferi nasıl?
S10 1091500.0                              Oyun sürekli çöküyor mu?
S11 1091500.0              Hemen mi alayım, indirimi mi bekleyeyim?
S12 1245620.0             Elden Ring yeni başlayan için çok mu zor?
S13 1245620.0                    Açık dünyayı keşfetmek keyifli mi?
S14       NaN                            Türkçe 

## 9. Test soruları (14)
Dosya: data/processed/kalite_testi_sorular.parquet (id, app_id, soru, olcut).
- 13 soru oyun filtreli (agent önce oyun_bul ile oyunu bulur), S14 filtresiz (bütün havuz).
- Zorlayıcılar: S10 çökme ≠ genel hata; S02/S11 günlük dil ("kasıyor", "indirim"); S14 havuzda az Türkçe yorum.
- Puanlama: `olcut`'a göre LLM hakem (havuzda zh/ru yorumlar var).
- Not: None yüzünden app_id float'a döndü → Int64'e çevrilecek.

In [1]:
import os, time
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
openai_istemci = OpenAI()
OLLAMA_URL = "http://localhost:11434/api/embed"

test_parcalar = pd.read_parquet("../data/processed/kalite_testi_parcalar.parquet")
test_sorulari = pd.read_parquet("../data/processed/kalite_testi_sorular.parquet")
test_sorulari["app_id"] = test_sorulari["app_id"].astype("Int64")
print("Parça:", len(test_parcalar), "| soru:", len(test_sorulari),
      "| app_id tipi:", test_sorulari["app_id"].dtype)
print()

def ollama_embed(model, metinler):
    cevap = requests.post(OLLAMA_URL, json={"model": model, "input": metinler}, timeout=600)
    cevap.raise_for_status()
    return np.array(cevap.json()["embeddings"], dtype=np.float32)

def openai_embed(metinler, model="text-embedding-3-small"):
    cevap = openai_istemci.embeddings.create(model=model, input=metinler)
    return np.array([d.embedding for d in cevap.data], dtype=np.float32)

MODELLER = {
    "bge-m3": lambda m: ollama_embed("bge-m3", m),
    "qwen3-4b": lambda m: ollama_embed("qwen3-embedding:4b", m),
    "openai-3-small": openai_embed,
}

deneme = [
    "Bu oyunda gece çok korkutucu.",
    "The night in this game is terrifying.",
    "Fiyatı çok pahalı.",
]
for ad, fonk in MODELLER.items():
    bas = time.time()
    v = fonk(deneme)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    benzerlik = v @ v.T
    print(f"{ad:<15} boyut: {v.shape[1]:>5}  süre: {time.time() - bas:5.2f} sn  "
          f"TR gece ↔ EN gece: {benzerlik[0, 1]:.3f}  TR gece ↔ TR fiyat: {benzerlik[0, 2]:.3f}")

Parça: 2447 | soru: 14 | app_id tipi: Int64

bge-m3          boyut:  1024  süre:  3.31 sn  TR gece ↔ EN gece: 0.933  TR gece ↔ TR fiyat: 0.620
qwen3-4b        boyut:  2560  süre:  4.14 sn  TR gece ↔ EN gece: 0.832  TR gece ↔ TR fiyat: 0.488
openai-3-small  boyut:  1536  süre:  2.61 sn  TR gece ↔ EN gece: 0.551  TR gece ↔ TR fiyat: 0.385


## 10. Modellere bağlantı + ısınma testi
`ollama_embed` (Windows Ollama, localhost:11434), `openai_embed` (.env'den anahtar), `MODELLER` sözlüğü.
Vektörler normalize → çarpım = kosinüs benzerliği (1 aynı anlam, 0 ilgisiz).

| Model | Boyut | TR gece ↔ EN gece | TR gece ↔ TR fiyat |
|---|---|---|---|
| bge-m3 | 1024 | 0,933 | 0,620 |
| qwen3-4b | 2560 | 0,832 | 0,488 |
| openai-3-small | 1536 | 0,551 | 0,385 |

- Üçü de diller arası anlamı yakalıyor (aynı anlam > aynı dil).
- Ölçekler farklı → modeller arası ham puan karşılaştırılmaz, her model kendi sıralamasıyla değerlendirilir.

In [2]:
from tqdm import tqdm

metinler = test_parcalar["embed_metin"].tolist()
PAKET = {"bge-m3": 32, "qwen3-4b": 32, "openai-3-small": 256}

vektorler = {}
for ad, fonk in MODELLER.items():
    bas = time.time()
    paketler = []
    for i in tqdm(range(0, len(metinler), PAKET[ad]), desc=ad):
        paketler.append(fonk(metinler[i:i + PAKET[ad]]))
    v = np.vstack(paketler)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    vektorler[ad] = v
    np.save(f"../data/processed/kalite_vektor_{ad}.npy", v)
    sure = time.time() - bas
    print(f"{ad:<15} {v.shape}  {sure:6.1f} sn  {len(metinler) / sure:6.1f} parça/sn")

print()
for m in requests.get("http://localhost:11434/api/ps").json()["models"]:
    print(f"{m['name']:<22} toplam: {m['size'] / 1e9:.2f} GB  ekran kartında: {m['size_vram'] / 1e9:.2f} GB")

bge-m3: 100%|██████████| 77/77 [00:37<00:00,  2.03it/s]


bge-m3          (2447, 1024)    37.9 sn    64.6 parça/sn


qwen3-4b: 100%|██████████| 77/77 [02:14<00:00,  1.75s/it]


qwen3-4b        (2447, 2560)   135.0 sn    18.1 parça/sn


openai-3-small: 100%|██████████| 10/10 [00:11<00:00,  1.17s/it]

openai-3-small  (2447, 1536)    11.7 sn   208.7 parça/sn

qwen3-embedding:4b     toplam: 3.43 GB  ekran kartında: 3.43 GB
bge-m3:latest          toplam: 0.69 GB  ekran kartında: 0.69 GB


## 11. 2.447 parçayı vektöre çevirme (hız + GPU)
Vektörler: data/processed/kalite_vektor_{model}.npy (normalize).

| Model | Boyut | Süre | Parça/sn | GPU |
|---|---|---|---|---|
| bge-m3 | 1024 | 37,9 sn | 64,6 | 0,69/0,69 GB ✓ |
| qwen3-4b | 2560 | 135,0 sn | 18,1 | 3,43/3,43 GB ✓ |
| openai-3-small | 1536 | 11,7 sn | 208,7 | (API) |

Tam veri kaba tahmini (~5 M parça): bge-m3 ~21 saat / ~20 GB, qwen3-4b ~3 gün / ~51 GB, 3-small birkaç saat / ~$5 / ~31 GB.
Kalite henüz ölçülmedi → sıradaki adım: 14 soru ile arama.

In [3]:
QWEN_TALIMAT = ("Instruct: Given a question about a video game, "
                "retrieve player reviews that answer the question\nQuery: ")
K = 5

def soru_metni(ad, soru):
    return QWEN_TALIMAT + soru if ad == "qwen3-4b" else soru

sonuclar = []
for ad, fonk in MODELLER.items():
    sv = fonk([soru_metni(ad, s) for s in test_sorulari["soru"]])
    sv = sv / np.linalg.norm(sv, axis=1, keepdims=True)

    for j, soru in test_sorulari.iterrows():
        if pd.isna(soru["app_id"]):
            maske = np.ones(len(test_parcalar), dtype=bool)
        else:
            maske = (test_parcalar["app_id"] == soru["app_id"]).to_numpy()
        adaylar = np.where(maske)[0]
        benzerlik = vektorler[ad][adaylar] @ sv[j]
        en_iyiler = np.argsort(-benzerlik)[:K]

        for sira, k in enumerate(en_iyiler, start=1):
            p = test_parcalar.iloc[adaylar[k]]
            sonuclar.append({
                "soru_id": soru["id"], "model": ad, "sira": sira,
                "parca_id": p["parca_id"], "benzerlik": round(float(benzerlik[k]), 3),
                "dil": p["dil"], "embed_metin": p["embed_metin"],
            })

sonuc = pd.DataFrame(sonuclar)
sonuc.to_parquet("../data/processed/kalite_testi_sonuclar.parquet", index=False)

print("İlk 5'in dilleri:")
print(sonuc.groupby(["soru_id", "model"])["dil"].apply(" ".join).unstack("model").to_string())
print()
havuz = sonuc.groupby("soru_id")["parca_id"].nunique()
print("Soru başına farklı parça (5 = üç model aynı parçaları buldu, 15 = hiç ortak yok):")
print(havuz.to_string())
print("Hakeme gidecek toplam:", havuz.sum())

for ad in MODELLER:
    print(f"\n######## S10 (Oyun sürekli çöküyor mu?) | {ad}")
    for _, r in sonuc[(sonuc["soru_id"] == "S10") & (sonuc["model"] == ad)].iterrows():
        print(f"\n[{r['sira']}] {r['parca_id']} ({r['dil']}, benzerlik {r['benzerlik']})")
        print(r["embed_metin"])

İlk 5'in dilleri:
model            bge-m3          openai-3-small        qwen3-4b
soru_id                                                        
S01      en en en ru en          en en de en pt  de en en en en
S02      zh zh en zh ru          tr ru tr tr zh  zh zh zh tr en
S03      en en en pt pt          tr tr en pt en  en en en pt en
S04      en en en en zh          tr tr tr tr tr  en zh zh en en
S05      en ja en en th          ru uk en en en  zh en zh en en
S06      en zh ru en zh          tr tr tr tr tr  en en en en en
S07      en en en en en          en fr en en en  en en en en en
S08      en en en en en          tr tr tr ru tr  tr en en en en
S09      en en pl en zh          tr pt zh en pl  zh zh zh zh en
S10      ru en en zh en          tr tr tr tr tr  zh en tr en zh
S11      en ru en zh en          tr tr tr tr tr  tr tr zh zh zh
S12      pt en en en en          en en en en pt  en en en pt en
S13      en ru es zh ko          tr tr ru zh zh  en en ru en es
S14      zh zh ko tr z

## 12. Arama sonuçları (14 soru × 3 model × ilk 5)
Dosya: data/processed/kalite_testi_sonuclar.parquet (210 satır). Qwen3 sorularına talimat öneki eklendi.

- **3-small dil yanlılığı:** havuzda %3 Türkçe olmasına rağmen S04/S06/S10/S11'de ilk 5'in hepsi Türkçe → anlam yerine dil eşliyor. Türkçe soru / %97 yabancı yorum senaryomuzda kritik zayıflık.
- S10 (çökme, genel hata sayılmaz) elle okuma: bge-m3 2/5, qwen3 2–3/5 (en iyi parça: "3 crashes to desktop"), 3-small 0–1/5.
- Modeller çoğunlukla farklı parça buluyor: 210 → 157 farklı parça → LLM hakemle puanlanacak.
- Kısa espri yorumlar (bge-m3: "like my body, broken") üste çıkabiliyor → meme riski.

In [4]:
import json

HAKEM_MODELI = "gpt-4.1-mini"
HAKEM_TALIMATI = """Sen bir arama sistemini değerlendiren hakemsin.
Sana bir kullanıcı sorusu, "doğru sayılır" ölçütü ve bir oyuncu yorumundan bir parça verilecek.
Parça herhangi bir dilde olabilir. Parça ölçütü karşılıyorsa ilgili=1, karşılamıyorsa ilgili=0 ver.
Sadece JSON döndür: {"ilgili": 0 veya 1, "gerekce": "Türkçe, tek cümle"}"""

def hakem(soru, olcut, parca):
    cevap = openai_istemci.chat.completions.create(
        model=HAKEM_MODELI,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": HAKEM_TALIMATI},
            {"role": "user", "content": f"Soru: {soru}\nÖlçüt: {olcut}\n\nParça:\n{parca}"},
        ],
    )
    return json.loads(cevap.choices[0].message.content)

ciftler = sonuc.drop_duplicates(["soru_id", "parca_id"]).merge(
    test_sorulari[["id", "soru", "olcut"]], left_on="soru_id", right_on="id"
)
print("Hakeme gidecek:", len(ciftler))

kararlar = []
for _, r in tqdm(ciftler.iterrows(), total=len(ciftler)):
    k = hakem(r["soru"], r["olcut"], r["embed_metin"])
    kararlar.append({"soru_id": r["soru_id"], "parca_id": r["parca_id"],
                     "ilgili": int(k["ilgili"]), "gerekce": k["gerekce"]})
kararlar = pd.DataFrame(kararlar)
kararlar.to_parquet("../data/processed/kalite_testi_hakem.parquet", index=False)

print("\nS10 hakem kararları:")
print(kararlar[kararlar["soru_id"] == "S10"][["parca_id", "ilgili", "gerekce"]].to_string(index=False))

puanli = sonuc.merge(kararlar, on=["soru_id", "parca_id"])
tablo = puanli.groupby(["soru_id", "model"])["ilgili"].sum().unstack("model")
print("\nSoru başına doğru parça (5 üzerinden):")
print(tablo.to_string())

ozet = pd.DataFrame({
    "isabet_%": (puanli.groupby("model")["ilgili"].mean() * 100).round(1),
    "en_az_1_dogru_soru": (tablo > 0).sum(),
    "toplam_dogru_/70": tablo.sum(),
})
print("\nÖzet:")
print(ozet.to_string())

Hakeme gidecek: 157


100%|██████████| 157/157 [02:38<00:00,  1.01s/it]


S10 hakem kararları:
       parca_id  ilgili                                                                                       gerekce
1091500_20669#0       1                                  Parçada oyunun belirli bir sahnede çökmesinden bahsediliyor.
  1091500_298#0       0                               Parçada oyunun çökmesinden veya kapanmasından bahsedilmemiştir.
 1091500_1031#0       0                               Parçada oyunun çökmesinden veya kapanmasından bahsedilmemiştir.
 1091500_3880#0       1                                    Parçada oyunun çökmesi ve kapanması açıkça belirtilmiştir.
1091500_22052#0       0                               Parçada oyunun çökmesinden veya kapanmasından bahsedilmemiştir.
 1091500_2116#0       1                Parçada oyunun masaüstüne atma şeklinde çökme yaşandığı açıkça belirtilmiştir.
1091500_24189#0       1        Parçada oyunun kapanması ve kapatıp açma gibi çökme belirtileri açıkça belirtilmiştir.
1091500_15255#0       0           

## 13. LLM hakem sonuçları (gpt-4.1-mini, 157 çift)
Hakem güvenilirliği: S10'da elle okumayla 12/12 aynı yön.

| Model | İsabet | Doğru /70 | En az 1 doğru soru |
|---|---|---|---|
| qwen3-4b | %82,9 | 58 | 14/14 |
| bge-m3 | %67,1 | 47 | 14/14 |
| openai-3-small | %44,3 | 31 | 11/14 |

- Qwen3 9 soruda önde, bge-m3 2, 3 berabere. En büyük fark S06 (soyut soru: 5 vs 2).
- 3-small elendi (dil yanlılığı: S04/S05/S06 = 0).
- S07 ve S14 herkes için zor (havuzda az ilgili yorum olabilir).
- Qwen3 bedeli: ~3,5× yavaş, 2,5× vektör boyutu, deploy'da 4B sorgu modeli.

In [5]:
def normalize(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def ara(pv, sv, k=K):
    satirlar = []
    for j, soru in test_sorulari.iterrows():
        if pd.isna(soru["app_id"]):
            adaylar = np.arange(len(test_parcalar))
        else:
            adaylar = np.where((test_parcalar["app_id"] == soru["app_id"]).to_numpy())[0]
        benzerlik = pv[adaylar] @ sv[j]
        for sira, kk in enumerate(np.argsort(-benzerlik)[:k], start=1):
            p = test_parcalar.iloc[adaylar[kk]]
            satirlar.append({"soru_id": soru["id"], "sira": sira,
                             "parca_id": p["parca_id"], "embed_metin": p["embed_metin"]})
    return pd.DataFrame(satirlar)

qwen_parca = vektorler["qwen3-4b"]
qwen_soru = MODELLER["qwen3-4b"]([soru_metni("qwen3-4b", s) for s in test_sorulari["soru"]])

ozet_satirlari = []
for boyut in [2560, 1024, 512, 256]:
    s = ara(normalize(qwen_parca[:, :boyut]), normalize(qwen_soru[:, :boyut]))

    yeni = s.merge(kararlar[["soru_id", "parca_id"]], how="left", indicator=True)
    yeni = yeni[yeni["_merge"] == "left_only"].merge(
        test_sorulari[["id", "soru", "olcut"]], left_on="soru_id", right_on="id"
    )
    for _, r in yeni.iterrows():
        k = hakem(r["soru"], r["olcut"], r["embed_metin"])
        kararlar.loc[len(kararlar)] = [r["soru_id"], r["parca_id"], int(k["ilgili"]), k["gerekce"]]

    if boyut == 2560:
        taban = set(zip(s["soru_id"], s["parca_id"]))
    ortak = len(taban & set(zip(s["soru_id"], s["parca_id"])))

    puanli_b = s.merge(kararlar, on=["soru_id", "parca_id"])
    ozet_satirlari.append({
        "boyut": boyut,
        "yeni_hakem": len(yeni),
        "2560_ile_ayni_%": round(ortak / len(s) * 100, 1),
        "isabet_%": round(puanli_b["ilgili"].mean() * 100, 1),
        "dogru_/70": int(puanli_b["ilgili"].sum()),
        "depolama_GB_5M_parca": round(5e6 * boyut * 4 / 1e9, 1),
    })

kararlar.to_parquet("../data/processed/kalite_testi_hakem.parquet", index=False)
print(pd.DataFrame(ozet_satirlari).to_string(index=False))
print("\nKarşılaştırma: bge-m3 %67,1 (47/70), 1024 boyut, ~20 GB")

 boyut  yeni_hakem  2560_ile_ayni_%  isabet_%  dogru_/70  depolama_GB_5M_parca
  2560           0            100.0      82.9         58                  51.2
  1024          12             80.0      82.9         58                  20.5
   512          13             62.9      72.9         51                  10.2
   256          17             62.9      71.4         50                   5.1

Karşılaştırma: bge-m3 %67,1 (47/70), 1024 boyut, ~20 GB


## 14. Qwen3 boyut küçültme (Matryoshka, yeniden embed yok)
| Boyut | 2560 ile aynı parça | İsabet | Doğru /70 | Depolama (5 M parça) |
|---|---|---|---|---|
| 2560 | %100 | %82,9 | 58 | 51,2 GB |
| **1024** | %80 | **%82,9** | **58** | **20,5 GB** |
| 512 | %62,9 | %72,9 | 51 | 10,2 GB |
| 256 | %62,9 | %71,4 | 50 | 5,1 GB |
Karşılaştırma: bge-m3 %67,1 (47/70), 1024 boyut, ~20 GB.

- 1024'te kalite kaybı yok; depolama bge-m3 ile aynı → Qwen3 1024 bge-m3'ten +16 puan.
- Kalan bedeller: hız (~3 gün / 5 M parça; boyut kesmek hızı değiştirmez), deploy'da 3,4 GB sorgu modeli.
- Model sabit: Ollama `qwen3-embedding:4b` (digest df5bd2e3c74c, Q4_K_M), sorgu öneki QWEN_TALIMAT.

## 15. KARAR: embedding modeli
**qwen3-embedding:4b** (Ollama, Windows, GPU), **1024 boyut** (ilk 1024 sayı + normalize), sorgularda `QWEN_TALIMAT` öneki.
- Neden: hakem testinde %82,9 (bge-m3 %67,1, OpenAI 3-small %44,3); 1024'te kalite kaybı yok, depolama bge-m3 ile aynı (~20 GB / 5 M parça).
- Bedel: tam veri ~3 gün (tek seferlik), deploy'da 3,4 GB sorgu modeli.
- Yerel model → emekliye ayrılma riski yok; sürüm sabit (digest df5bd2e3c74c).
- OpenAI anahtarı hakem için kalıyor (08'de de kullanılacak).

In [7]:
import chromadb

istemci = chromadb.PersistentClient(path="../data/chroma")
if "yorumlar_test" in [c.name for c in istemci.list_collections()]:
    istemci.delete_collection("yorumlar_test")
koleksiyon = istemci.create_collection(
    name="yorumlar_test",
    configuration={"hnsw": {"space": "cosine"}},
    embedding_function=None,
)

BOYUT = 1024
parca_v = normalize(qwen_parca[:, :BOYUT])
soru_v = normalize(qwen_soru[:, :BOYUT])

META_SUTUNLARI = ["review_id", "app_id", "olumlu", "dil", "tarih", "faydali_oy", "parca_no", "parca_sayisi"]
metalar = [
    {k: (v.item() if hasattr(v, "item") else v) for k, v in m.items()}
    for m in test_parcalar[META_SUTUNLARI].to_dict("records")
]

PAKET_C = 1000
for i in range(0, len(test_parcalar), PAKET_C):
    koleksiyon.add(
        ids=test_parcalar["parca_id"].iloc[i:i + PAKET_C].tolist(),
        embeddings=parca_v[i:i + PAKET_C],
        documents=test_parcalar["embed_metin"].iloc[i:i + PAKET_C].tolist(),
        metadatas=metalar[i:i + PAKET_C],
    )
print("Koleksiyondaki parça:", koleksiyon.count())

j = test_sorulari.index[test_sorulari["id"] == "S10"][0]
cevap = koleksiyon.query(query_embeddings=soru_v[j:j + 1], n_results=5,
                         where={"app_id": 1091500})
chroma_ids = cevap["ids"][0]
numpy_ids = ara(parca_v, soru_v).query("soru_id == 'S10'")["parca_id"].tolist()
print("\nChroma:", chroma_ids)
print("numpy :", numpy_ids)
print("Aynı mı:", chroma_ids == numpy_ids)
print("Mesafeler:", [round(d, 3) for d in cevap["distances"][0]])

cevap2 = koleksiyon.query(
    query_embeddings=soru_v[j:j + 1], n_results=3,
    where={"$and": [{"app_id": 1091500}, {"olumlu": 0}]},
)
print("\n=== S10, sadece olumsuz Cyberpunk yorumları ===")
for pid, meta, metin in zip(cevap2["ids"][0], cevap2["metadatas"][0], cevap2["documents"][0]):
    print(f"\n{pid} | olumlu={meta['olumlu']} | {meta['dil']} | {meta['tarih']}")
    print(metin)

Koleksiyondaki parça: 2447

Chroma: ['1091500_24189#0', '1091500_3880#0', '1091500_2116#0', '1091500_6268#0', '1091500_22052#0']
numpy : ['1091500_24189#0', '1091500_3880#0', '1091500_2116#0', '1091500_6268#0', '1091500_22052#0']
Aynı mı: True
Mesafeler: [0.331, 0.333, 0.349, 0.351, 0.352]

=== S10, sadece olumsuz Cyberpunk yorumları ===

1091500_3880#0 | olumlu=0 | zh | 2023-06-07
两年多没玩重新下回来就崩溃闪退打不开，这两年修了个啥还不如以前

1091500_2116#0 | olumlu=0 | en | 2020-12-09
Edit Jan 2022: I've put about a dozen hours into this on another platform after picking the game up for $20. The game is MUCH more stable than it used to be. That said, I've had three crashes to desktop within those 12 hours. I'm really enjoying my time with the game, but given those crashes and glitches, I would have been miffed if I'd paid full price. ----------------------------------Original Review:I'm really looking forward to playing this game once the bugs are addressed. But the bugs even in the initial hour are substantial: 

## 16. Chroma test koleksiyonu
`chromadb.PersistentClient("../data/chroma")`, koleksiyon `yorumlar_test`: kosinüs, embedding_function=None (vektörleri biz veriyoruz), Qwen3 1024.
Her parça: id = parca_id, vektör, metin, metadata (review_id, app_id, olumlu, dil, tarih, faydali_oy, parca_no, parca_sayisi).

- 2.447 parça eklendi; S10 Chroma sonucu numpy aramasıyla birebir aynı ✓. Mesafe = 1 − benzerlik (0,33–0,35).
- `where={"$and": [{"app_id": ...}, {"olumlu": 0}]}` filtresi çalışıyor.
- Not: `tarih` ilk yazılma tarihi; yorum sonradan düzenlenmiş olabilir ("Edit Jan 2022") → 07 agent kuralı.

In [8]:
import sqlite3
con = sqlite3.connect("file:../data/processed/steam.db?mode=ro", uri=True)

def katman_karma(katman, n):
    if len(katman) <= n:
        return katman
    n_faydali = n // 2
    faydali = katman.nlargest(n_faydali, "faydali_oy")
    kalan = katman.drop(faydali.index)
    return pd.concat([faydali, kalan.sample(n - n_faydali, random_state=42)])

def karma_sec(y, tavan, olumlu_oran):
    if len(y) <= tavan:
        return y
    if pd.isna(olumlu_oran):
        olumlu_oran = y["olumlu"].mean()
    olumlular = y[y["olumlu"] == 1]
    olumsuzlar = y[y["olumlu"] == 0]
    n_olumlu = min(round(tavan * olumlu_oran), len(olumlular))
    n_olumsuz = min(tavan - n_olumlu, len(olumsuzlar))
    return pd.concat([katman_karma(olumlular, n_olumlu), katman_karma(olumsuzlar, n_olumsuz)])

def parca_sayisi(token):
    return np.where(token <= 575, 1, np.ceil((token - 575) / 448) + 1).astype(int)

bas = time.time()
meta = pd.read_sql(
    "SELECT rowid AS satir, app_id, olumlu, faydali_oy, uzunluk "
    "FROM yorumlar WHERE chroma_uygun = 1", con)
oranlar = pd.read_sql("SELECT app_id, olumlu_oran FROM oyunlar", con).set_index("app_id")["olumlu_oran"]
print(f"Okundu: {len(meta):,} yorum, {time.time() - bas:.0f} sn, "
      f"bellek {meta.memory_usage().sum() / 1e9:.2f} GB")

QWEN_HIZ = 18.1   # parça/sn, 12. hücreden
satirlar = []
for tavan in [500, 1000]:
    bas = time.time()
    secim = pd.concat([
        karma_sec(grup, tavan, oranlar.get(app_id))
        for app_id, grup in meta.groupby("app_id")
    ])
    token = secim["uzunluk"] / 3.42
    parca = parca_sayisi(token.to_numpy()).sum()
    secim[["satir", "app_id"]].to_parquet(f"../data/processed/secim_tavan{tavan}.parquet", index=False)
    satirlar.append({
        "tavan": tavan,
        "yorum_M": round(len(secim) / 1e6, 2),
        "olumlu_%": round(secim["olumlu"].mean() * 100, 1),
        "token_M": round(token.sum() / 1e6),
        "parca_M": round(parca / 1e6, 2),
        "qwen_saat": round(parca / QWEN_HIZ / 3600, 1),
        "vektor_GB": round(parca * 1024 * 4 / 1e9, 1),
        "secim_sn": round(time.time() - bas),
    })

print(pd.DataFrame(satirlar).to_string(index=False))

Okundu: 21,846,768 yorum, 37 sn, bellek 0.87 GB
 tavan  yorum_M  olumlu_%  token_M  parca_M  qwen_saat  vektor_GB  secim_sn
   500     4.05      82.0      489     4.31       66.1       17.6        11
  1000     5.70      82.3      655     6.03       92.6       24.7         8


## 17. Tam veri: tavan 500 vs 1000 (gerçek karma seçim)
Kaynak: yorumlar (chroma_uygun=1) 21.846.768 satır, 37 sn, 0,87 GB. Token ≈ uzunluk / 3,42.
Seçim listeleri: data/processed/secim_tavan500.parquet, secim_tavan1000.parquet (satir = rowid, app_id).

| Tavan | Yorum | Olumlu | Token | Parça | Qwen3 süre (18,1/sn) | Vektör |
|---|---|---|---|---|---|---|
| 500 | 4,05 M | %82,0 | 489 M | 4,31 M | 66,1 saat | 17,6 GB |
| 1000 | 5,70 M | %82,3 | 655 M | 6,03 M | 92,6 saat | 24,7 GB |

- Karma uzun yorumları seçtiği için ortalama ~121 token/yorum (genel ortalama 88).
- Parçalama tam veride +%6 (test havuzunda +%22; havuz sadece dev oyunlardı).
- 1000: +%41 yorum, +26,5 saat, +7 GB; ek yorumlar sadece en büyük ~2.700 oyuna gidiyor.
- Hız test havuzunda ölçüldü (uzun parçalar) → gerçek hız muhtemelen daha yüksek; karar hız testinden sonra.

In [9]:
from concurrent.futures import ThreadPoolExecutor

secim500 = pd.read_parquet("../data/processed/secim_tavan500.parquet")
ornek_satir = secim500["satir"].sample(2000, random_state=1).tolist()
hiz_ornek = pd.read_sql(
    f"SELECT metin FROM yorumlar WHERE rowid IN ({','.join(map(str, ornek_satir))})", con)
hiz_metinler = hiz_ornek["metin"].str[:1750].tolist()
print("Örnek:", len(hiz_metinler), "| ort. karakter:", round(np.mean([len(m) for m in hiz_metinler])))

def hiz_olc(paket, eszamanli):
    paketler = [hiz_metinler[i:i + paket] for i in range(0, len(hiz_metinler), paket)]
    bas = time.time()
    with ThreadPoolExecutor(max_workers=eszamanli) as havuz:
        list(havuz.map(lambda p: ollama_embed("qwen3-embedding:4b", p), paketler))
    return len(hiz_metinler) / (time.time() - bas)

ollama_embed("qwen3-embedding:4b", ["ısınma"])

satirlar = []
for paket in [32, 128]:
    for eszamanli in [1, 4]:
        h = hiz_olc(paket, eszamanli)
        satirlar.append({
            "paket": paket, "eszamanli": eszamanli, "parca_sn": round(h, 1),
            "tavan500_saat": round(4.31e6 / h / 3600, 1),
            "tavan1000_saat": round(6.03e6 / h / 3600, 1),
        })
        print(satirlar[-1])

print()
print(pd.DataFrame(satirlar).to_string(index=False))

Örnek: 2000 | ort. karakter: 343
{'paket': 32, 'eszamanli': 1, 'parca_sn': 27.4, 'tavan500_saat': 43.6, 'tavan1000_saat': 61.0}
{'paket': 32, 'eszamanli': 4, 'parca_sn': 27.2, 'tavan500_saat': 44.1, 'tavan1000_saat': 61.7}
{'paket': 128, 'eszamanli': 1, 'parca_sn': 25.9, 'tavan500_saat': 46.2, 'tavan1000_saat': 64.7}


HTTPError: 400 Client Error: Bad Request for url: http://localhost:11434/api/embed

## 18. Hız testi (Qwen3, gerçek veri 2.000 yorum, ort. 343 kar.)
| Paket | Eşzamanlı | Parça/sn | Tavan 500 | Tavan 1000 |
|---|---|---|---|---|
| 32 | 1 | **27,4** | 43,6 saat | 61,0 saat |
| 32 | 4 | 27,2 | 44,1 saat | 61,7 saat |
| 128 | 1 | 25,9 | 46,2 saat | 64,7 saat |
| 128 | 4 | 400 Bad Request | – | – |

- Gerçek veri test havuzundan hızlı (27,4 vs 18,1; yorumlar daha kısa).
- Paralel istek kazanç sağlamıyor → GPU zaten dolu. En iyi: paket 32, tek istek.
- Büyük yük 400 hatası verdi → tam veri döngüsüne hata/yeniden deneme koruması + kaldığı yerden devam.

In [10]:
TOKEN_HIZ = 2400   # token/sn: deneme bloğu (~21 parça/sn) ile tavan 500 tahmininden

satirlar = []
for tavan in [200, 300, 500]:
    secim = pd.concat([
        karma_sec(grup, tavan, oranlar.get(app_id))
        for app_id, grup in meta.groupby("app_id")
    ])
    secim[["satir", "app_id"]].to_parquet(f"../data/processed/secim_tavan{tavan}.parquet", index=False)
    token = secim["uzunluk"].to_numpy() / 3.42

    for maks_parca in [None, 2]:
        t = token if maks_parca is None else np.minimum(token, 512 + (maks_parca - 1) * 448)
        satirlar.append({
            "tavan": tavan,
            "maks_parca": maks_parca or "sınırsız",
            "yorum_M": round(len(secim) / 1e6, 2),
            "token_M": round(t.sum() / 1e6),
            "parca_M": round(parca_sayisi(t).sum() / 1e6, 2),
            "tahmini_saat": round(t.sum() / TOKEN_HIZ / 3600, 1),
        })

print(pd.DataFrame(satirlar).to_string(index=False))

 tavan maks_parca  yorum_M  token_M  parca_M  tahmini_saat
   200   sınırsız     2.46      317     2.63          36.7
   200          2     2.46      295     2.57          34.2
   300   sınırsız     3.09      387     3.30          44.8
   300          2     3.09      361     3.22          41.8
   500   sınırsız     4.05      489     4.31          56.6
   500          2     4.05      457     4.21          52.9


In [11]:
KUCUK = "qwen3-embedding:0.6b"

k_parca = np.vstack([ollama_embed(KUCUK, metinler[i:i + 32]) for i in range(0, len(metinler), 32)])
k_parca = normalize(k_parca[:, :1024])
k_soru = normalize(ollama_embed(KUCUK, [QWEN_TALIMAT + s for s in test_sorulari["soru"]])[:, :1024])
print("Boyut:", k_parca.shape)

s = ara(k_parca, k_soru)
yeni = s.merge(kararlar[["soru_id", "parca_id"]], how="left", indicator=True)
yeni = yeni[yeni["_merge"] == "left_only"].merge(
    test_sorulari[["id", "soru", "olcut"]], left_on="soru_id", right_on="id")
for _, r in yeni.iterrows():
    k = hakem(r["soru"], r["olcut"], r["embed_metin"])
    kararlar.loc[len(kararlar)] = [r["soru_id"], r["parca_id"], int(k["ilgili"]), k["gerekce"]]
kararlar.to_parquet("../data/processed/kalite_testi_hakem.parquet", index=False)

p = s.merge(kararlar, on=["soru_id", "parca_id"])
tablo_k = p.groupby("soru_id")["ilgili"].sum()
print(f"Yeni hakem: {len(yeni)} | isabet: %{p['ilgili'].mean() * 100:.1f} | "
      f"doğru: {p['ilgili'].sum()}/70 | en az 1 doğru: {(tablo_k > 0).sum()}/14")
print("Karşılaştırma: 4b-1024 %82,9 (58/70), bge-m3 %67,1 (47/70)")
print("\nSoru başına doğru (0.6b):")
print(tablo_k.to_string())

ollama_embed(KUCUK, ["ısınma"])
bas = time.time()
for i in range(0, len(hiz_metinler), 32):
    ollama_embed(KUCUK, hiz_metinler[i:i + 32])
hiz = len(hiz_metinler) / (time.time() - bas)
gercekci = hiz * 21 / 27.4
print(f"\nHız: {hiz:.1f} parça/sn (4b: 27,4) | script içinde ~{gercekci:.0f} parça/sn | "
      f"tavan 500 ≈ {4.31e6 / gercekci / 3600:.1f} saat")

Boyut: (2447, 1024)
Yeni hakem: 23 | isabet: %70.0 | doğru: 49/70 | en az 1 doğru: 14/14
Karşılaştırma: 4b-1024 %82,9 (58/70), bge-m3 %67,1 (47/70)

Soru başına doğru (0.6b):
soru_id
S01    4
S02    4
S03    5
S04    2
S05    1
S06    4
S07    1
S08    5
S09    5
S10    4
S11    2
S12    5
S13    4
S14    3

Hız: 77.0 parça/sn (4b: 27,4) | script içinde ~59 parça/sn | tavan 500 ≈ 20.3 saat


### Bozulan Chroma'nın kurtarılması

- **Sorun:** Tam veri Chroma'ya yazılırken PC kapatıldı. HNSW indeksi kuyruğun gerisinde kaldı (2.000 kayıt), `count()` bile "compaction" hatası verdi.
- **Çözüm:**
  1. `data/chroma` klasörü yedeklendi (`data/chroma_yedek`, 10 GB).
  2. `chroma.sqlite3` içindeki `embeddings_queue` tablosundan HNSW'ye hiç yazılmamış 2.000 kayıt silindi. Bunlar blok 52'nin yarım kalan yazmasıydı.
  3. Chroma açıldı (count 1.124.217), `get` ve `query` doğru çalıştı.
  4. Parçalar 20.000'erlik sayfalarla okundu, yeni script'in dosyalarında zaten bulunanlar atlandı, kalanlar `kurtarilan_XXX.parquet` + `.npy` olarak yazıldı.
- **Sonuç:** 1.105.217 parça (blok 1–51) kurtarıldı, tekrar ve çakışma yok. Toplam 2.644.490 parça dosyada.
- **Ders:** Chroma'ya uzun süreli, kesintili yazma yapma. Önce vektörleri dosyaya yaz (kapatma güvenli), sonra Chroma'ya tek seferde ve kesintisiz yükle.

### Yorum koleksiyonunun toplu yüklenmesi

- **Script:** `src/chroma_yukle.py`. Kaynak 129 dosya: 56 `kurtarilan_*` (blok 1–51) + 73 `blok_*` (blok 52–124).
- **Hedef:** yeni ve temiz klasör `data/chroma_v2`, koleksiyon `yorumlar`, cosine mesafe, `embedding_function=None` (vektörleri biz veriyoruz).
- **Yöntem:** Her dosya 5.461'lik paketlerle `upsert` edildi. Biten dosyalar `yukleme_ilerleme.json`'a yazıldı, yani script kaldığı yerden devam edebilir. Metadata türleri tek tipe çevrildi (sayılar int, diğerleri str) ki filtreler tutarlı çalışsın.
- **Sonuç:** 2.644.490 parça, beklenenle fark 0. Test araması doğru. WSL RAM'i (16 GB) yetti, süre ~2 saat.
- **Koleksiyon özeti:** qwen3-embedding:4b, 1024 boyut, oyun başına en fazla 200 yorum (karma seçim), uzun yorumlar 512/64 token parçalı. Metadata: review_id, app_id, olumlu, dil, tarih, faydali_oy, parca_no, parca_sayisi.

In [2]:
import sqlite3
import pandas as pd
from transformers import AutoTokenizer

con = sqlite3.connect("file:../data/processed/steam.db?mode=ro", uri=True)
oyun = pd.read_sql("SELECT app_id, ad, kisa_aciklama, oyun_hakkinda, yorum_var, oy_sayisi, iskelet FROM oyunlar", con)
etiket = pd.read_sql("SELECT app_id, etiket, sira FROM oyun_etiket", con)
ilk10 = (etiket.sort_values(["app_id", "sira"]).groupby("app_id").head(10)
         .groupby("app_id")["etiket"].agg(", ".join).rename("etiketler"))
oyun = oyun.merge(ilk10, on="app_id", how="left")

def metin_mi(x):
    return isinstance(x, str) and x.strip() != ""

def oyun_metni(r):
    parcalar = [r.ad]
    if metin_mi(r.kisa_aciklama):
        parcalar.append(r.kisa_aciklama.strip())
    if metin_mi(r.oyun_hakkinda):
        parcalar.append(r.oyun_hakkinda.strip()[:2000])
    if metin_mi(r.etiketler):
        parcalar.append("Etiketler: " + r.etiketler)
    return "\n".join(parcalar)

oyun["metin"] = [oyun_metni(r) for r in oyun.itertuples()]
oyun["spam"] = oyun["metin"].str.contains("其他游戏推荐|友情链接")
print("oyun:", len(oyun), "| etiketsiz:", oyun["etiketler"].isna().sum(),
      "| açıklamasız:", (~oyun["kisa_aciklama"].apply(metin_mi) & ~oyun["oyun_hakkinda"].apply(metin_mi)).sum(),
      "| spam işaretli:", oyun["spam"].sum())

tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
ornek = oyun.sample(5000, random_state=42).copy()
ornek["token"] = [len(x) for x in tok(ornek["metin"].tolist(), add_special_tokens=False)["input_ids"]]
print("örnek ort. karakter:", round(ornek["metin"].str.len().mean()), "| ort. token:", round(ornek["token"].mean()),
      "| en uzun token:", ornek["token"].max())

TOKEN_HIZ = 2400
secenekler = {
    "tümü": oyun["app_id"] > 0,
    "iskelet değil": oyun["iskelet"] == 0,
    "yorumlu (CSV'li)": oyun["yorum_var"] == 1,
    "yorumlu veya oy>=100": (oyun["yorum_var"] == 1) | (oyun["oy_sayisi"] >= 100),
}
satirlar = []
for ad, maske in secenekler.items():
    ort_token = ornek.loc[ornek["app_id"].isin(oyun.loc[maske, "app_id"]), "token"].mean()
    toplam = maske.sum() * ort_token
    satirlar.append({"seçenek": ad, "oyun": int(maske.sum()), "ort_token": round(ort_token),
                     "toplam_M_token": round(toplam / 1e6, 1), "tahmini_dk": round(toplam / TOKEN_HIZ / 60)})
display(pd.DataFrame(satirlar))

display(oyun.loc[oyun["ad"].isin(["Starfield", "It Takes Two", "Hollow Knight"]),
                 ["app_id", "ad", "yorum_var", "oy_sayisi", "iskelet", "etiketler"]])
print(oyun.loc[oyun["ad"] == "Starfield", "metin"].iloc[0])

oyun: 64948 | etiketsiz: 20717 | açıklamasız: 5051 | spam işaretli: 0
örnek ort. karakter: 1300 | ort. token: 326 | en uzun token: 1576


,seçenek,oyun,ort_token,toplam_M_token,tahmini_dk
0,tümü,64948,326,21.2,147
1,iskelet değil,60131,353,21.3,148
2,yorumlu (CSV'li),22898,396,9.1,63
3,yorumlu veya oy>=100,23218,396,9.2,64


,app_id,ad,yorum_var,oy_sayisi,iskelet,etiketler
12564,1426210,It Takes Two,0,180116,0,"Co-op, Multiplayer, Split Screen, Local Co-Op,..."
22864,1716740,Starfield,0,160947,0,"Space, Singleplayer, Open World, RPG, Sci-fi, ..."


Starfield
Starfield is the first new universe in 25 years from Bethesda Game Studios, the award-winning creators of The Elder Scrolls V: Skyrim and Fallout 4.
Starfield is the first new universe in over 25 years from Bethesda Game Studios, the award-winning creators of The Elder Scrolls V: Skyrim and Fallout 4. In this next generation role-playing game set amongst the stars, create any character you want and explore with unparalleled freedom as you embark on an epic journey to answer humanity’s greatest mystery. In the year 2330, humanity has ventured beyond our solar system, settling new planets, and living as a spacefaring people. You will join Constellation – the last group of space explorers seeking rare artifacts throughout the galaxy – and navigate the vast expanse of space in Bethesda Game Studios’ biggest and most ambitious game. Tell Your story In Starfield the most important story is the one you tell with your character. Start your journey by customizing your appearance and d

### Oyun koleksiyonu

- **Hangi oyunlar (KARAR):** `yorum_var = 1 OR oy_sayisi >= 100` → 23.218 oyun. Sadece CSV'liler alınsaydı Starfield / It Takes Two gibi 320 popüler oyun aramada hiç çıkmazdı. Hepsi (64.948) ~2,5 saat sürerdi ve fazladan gelen oyunlar az oylu ve yorumsuzdu.
- **Metin:** ad + kısa açıklama + about'un ilk 2.000 karakteri + ilk 10 etiket. Ortalama 326 token, en uzun 1.576 → parçalama gerekmedi. Oyunların %32'sinde etiket yok.
- **Script:** `src/oyun_embed.py`. Başta GPU kontrolü yapılır, vektörler önce dosyaya yazılır (24 blok), sonra `data/chroma_v2` içine `oyunlar` koleksiyonu olarak yüklenir. ~36 dk.
- **Metadata:** app_id, ad, yorum_var, oy_sayisi, yetiskin.
- **Test:** "uzayda geçen, gezegen keşfedilen açık dünya rol yapma oyunu" → Starfield (0,397), helionaut, Far Horizon, Sable, Lightyear Frontier.
- **06'ya not:** az oylu oyunlar üste çıkabiliyor → oy filtresi. Sorgu öneki yorumlar için yazılmıştı, oyunlar için ayrı bir öneki denemek gerekiyor.